# D8 Baselines on GPU — Phase B+ Deep Choice Paper

Re-run gravity / radiation / Deep Gravity baselines on Colab GPU so the Deep Gravity result actually converges. Mirrors `experiments/paper_a/baselines_spatial_holdout.py`.

**Steps**
1. Runtime → Change runtime type → **T4 GPU** (free) or A100 (Pro).
2. Run cell 1 to verify GPU.
3. Upload **4 data files** in cell 2 (use the file picker that pops up):
   - `demo_cache_slim.npz` (~2 MB, slim version of demo_cache)
   - `paperA_v23_aux.npz` (~24 MB)
   - `car_freeflow_t_ij.npy` (~12 MB)
   - `grid_static_features.csv` (~500 KB)
4. Run cells 3-7 in order. Total ~30-50 min on T4, ~15 min on A100.
5. Cell 8 packages results to download.

**What you get back**: `baselines_spatial_holdout.csv` and `baselines_spatial_holdout.json` — drop into `evaluation_outputs/paper_a/` locally.

In [ ]:
# === Cell 1: verify GPU ===
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))
    print('VRAM GB:', torch.cuda.get_device_properties(0).total_memory / 1e9)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Will use:', DEVICE)

In [ ]:
# === Cell 2: upload data files ===
from google.colab import files
print('Pick the 4 files: demo_cache_slim.npz, paperA_v23_aux.npz, car_freeflow_t_ij.npy, grid_static_features.csv')
uploaded = files.upload()
import os
for fn in ['demo_cache_slim.npz', 'paperA_v23_aux.npz', 'car_freeflow_t_ij.npy', 'grid_static_features.csv']:
    assert os.path.exists(fn), f'missing: {fn}'
print('All 4 files present.')

In [ ]:
# === Cell 3: load data + build masks ===
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import time, json

HELDOUT_BOROUGHS = ['Westminster', 'Hackney', 'Brent', 'Bromley']

cache = np.load('demo_cache_slim.npz', allow_pickle=True)
F_ij = torch.tensor(cache['F_ij_t'].sum(0), dtype=torch.float32)
t_ij = torch.tensor(np.load('car_freeflow_t_ij.npy').astype(np.float32), dtype=torch.float32)
static = torch.tensor(cache['static_features'], dtype=torch.float32)
coords = cache['coords_bng']
diff = coords[:, None, :] - coords[None, :, :]
d_km = np.sqrt((diff ** 2).sum(-1)) / 1000.0
log_d = torch.tensor(np.log1p(d_km), dtype=torch.float32)

boroughs = cache['boroughs']
gbi = cache['grid_borough_idx']
name_to_idx = {b: i for i, b in enumerate(boroughs)}
held_idxs = [name_to_idx[n] for n in HELDOUT_BOROUGHS]
val_mask = torch.tensor(np.isin(gbi, held_idxs))
train_mask = ~val_mask
print(f'train: {int(train_mask.sum())}  val: {int(val_mask.sum())}  total: {len(gbi)}')

df_grid = pd.read_csv('grid_static_features.csv')
grid_ids = cache['grid_ids'].tolist()
df_grid = df_grid.set_index('grid_id').reindex(grid_ids)
D_j = torch.tensor(df_grid['total_employment'].fillna(0.0).to_numpy(), dtype=torch.float32)
m_i = df_grid['population'].fillna(0.0).to_numpy().astype(np.float64)
n_j = df_grid['total_employment'].fillna(0.0).to_numpy().astype(np.float64)
F_obs_np = F_ij.numpy()
val_mask_np = val_mask.numpy().astype(bool)

def cpc(F_obs, F_pred, mask):
    obs = F_obs[mask]; pr = F_pred[mask]
    return float(2.0 * np.minimum(pr, obs).sum() / max(pr.sum() + obs.sum(), 1.0))

In [ ]:
# === Cell 4: fit Gravity (Wilson, softmax MLE) ===
def fit_gravity_softmax(F_ij, D_j, t_ij, log_d_ij, train_mask, epochs=400, lr=0.05):
    N = F_ij.shape[0]
    log_D = torch.log(D_j.clamp(min=1.0))
    log_D_b = log_D.unsqueeze(0).expand(N, N)
    raw_alpha = torch.tensor(0.0, requires_grad=True)
    raw_beta = torch.tensor(np.log(np.exp(0.05) - 1.0), requires_grad=True)
    raw_gamma = torch.tensor(np.log(np.exp(1.0) - 1.0), requires_grad=True)
    opt = torch.optim.Adam([raw_alpha, raw_beta, raw_gamma], lr=lr)
    O_i = F_ij.sum(dim=1, keepdim=True)
    train_idx = train_mask.bool()
    F_train = F_ij[train_idx]
    last = {}
    for ep in range(epochs):
        opt.zero_grad()
        alpha = F.softplus(raw_alpha); beta = F.softplus(raw_beta); gamma = F.softplus(raw_gamma)
        V = alpha * log_D_b - beta * t_ij - gamma * log_d_ij
        V = V.clone(); V.fill_diagonal_(-1e9)
        log_p = F.log_softmax(V, dim=1)
        loss = -(F_train * log_p[train_idx]).sum() / max(F_train.sum().item(), 1.0)
        loss.backward(); opt.step()
    with torch.no_grad():
        alpha = F.softplus(raw_alpha); beta = F.softplus(raw_beta); gamma = F.softplus(raw_gamma)
        V = alpha * log_D_b - beta * t_ij - gamma * log_d_ij
        V = V.clone(); V.fill_diagonal_(-1e9)
        P = F.softmax(V, dim=1)
        F_pred = O_i * P
    return F_pred, {'alpha': float(alpha), 'beta': float(beta), 'gamma': float(gamma), 'loss': float(loss)}

t0 = time.time()
F_pred_grav, info_grav = fit_gravity_softmax(F_ij, D_j, t_ij, log_d, train_mask)
grav_cpc = cpc(F_obs_np, F_pred_grav.numpy(), val_mask_np)
print(f'Gravity val_CPC(spatial) = {grav_cpc:.4f}  alpha={info_grav["alpha"]:.3f} beta={info_grav["beta"]:.4f} gamma={info_grav["gamma"]:.3f}  t={time.time()-t0:.1f}s')

In [ ]:
# === Cell 5: fit Radiation (Simini 2012 parameter-free) ===
def fit_radiation(F_ij, m_i, n_j, coords):
    N = len(m_i)
    diff = coords[:, None, :] - coords[None, :, :]
    d_ij = np.sqrt((diff ** 2).sum(axis=-1)) / 1000.0
    order = np.argsort(d_ij, axis=1)
    n_sorted = n_j[order]
    cum = np.cumsum(n_sorted, axis=1)
    s_in_order = cum - n_sorted
    inv_order = np.argsort(order, axis=1)
    s = np.take_along_axis(s_in_order, inv_order, axis=1)
    s = np.maximum(s, 0.0); np.fill_diagonal(s, 0.0)
    M = m_i[:, None]; Nj = n_j[None, :]
    denom = (M + s) * (M + Nj + s)
    denom = np.where(denom <= 0, np.nan, denom)
    prob = (M * Nj) / denom
    prob = np.nan_to_num(prob, nan=0.0); np.fill_diagonal(prob, 0.0)
    rs = prob.sum(axis=1, keepdims=True)
    rs = np.where(rs <= 0, 1.0, rs)
    prob = prob / rs
    O_i = F_ij.sum(dim=1).numpy()
    return torch.tensor(O_i[:, None] * prob, dtype=torch.float32)

t0 = time.time()
F_pred_rad = fit_radiation(F_ij, m_i, n_j, coords)
rad_cpc = cpc(F_obs_np, F_pred_rad.numpy(), val_mask_np)
print(f'Radiation val_CPC(spatial) = {rad_cpc:.4f}  t={time.time()-t0:.1f}s')

In [ ]:
# === Cell 6: Deep Gravity model + train (GPU-aware, Simini 2021 spirit) ===
class _DeepGravityMLP(nn.Module):
    def __init__(self, feat_dim, hidden=128, n_layers=5):
        super().__init__()
        layers = []
        d_in = 2 * feat_dim + 1
        for _ in range(n_layers):
            layers += [nn.Linear(d_in, hidden), nn.ReLU()]
            d_in = hidden
        layers += [nn.Linear(d_in, 1)]
        self.net = nn.Sequential(*layers)
    def forward_block(self, X_i_block, X_j, log_d_block):
        B = X_i_block.shape[0]; N = X_j.shape[0]
        Xi_b = X_i_block.unsqueeze(1).expand(B, N, -1)
        Xj_b = X_j.unsqueeze(0).expand(B, N, -1)
        d_b = log_d_block.unsqueeze(-1)
        feat = torch.cat([Xi_b, Xj_b, d_b], dim=-1)
        return self.net(feat).squeeze(-1)

def fit_deep_gravity(F_ij, static, log_d_ij, train_mask, val_mask,
                     epochs=1000, lr=1e-3, hidden=128, n_layers=5,
                     seed=0, batch_size=128, early_stop_patience=40,
                     device='cpu', verbose_every=50):
    torch.manual_seed(seed); np.random.seed(seed)
    dev = torch.device(device)
    F_ij_d = F_ij.to(dev); static_d = static.to(dev); log_d_ij_d = log_d_ij.to(dev)
    train_mask_d = train_mask.to(dev); val_mask_d = val_mask.to(dev)
    N, Fdim = static.shape
    model = _DeepGravityMLP(feat_dim=Fdim, hidden=hidden, n_layers=n_layers).to(dev)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    train_idx = torch.where(train_mask_d.bool())[0]
    val_idx = torch.where(val_mask_d.bool())[0]
    F_train_total = F_ij_d[train_idx].sum().item()
    F_val_total = F_ij_d[val_idx].sum().item()
    O_i = F_ij_d.sum(dim=1, keepdim=True)
    best_val = float('inf'); best_state = None; no_improve = 0
    for ep in range(epochs):
        model.train()
        perm = train_idx[torch.randperm(len(train_idx), device=dev)]
        ep_loss = 0.0; n_b = 0
        for b0 in range(0, len(perm), batch_size):
            bo = perm[b0:b0+batch_size]
            opt.zero_grad()
            h = model.forward_block(static_d[bo], static_d, log_d_ij_d[bo])
            mask_diag = torch.zeros_like(h)
            for k, oid in enumerate(bo):
                mask_diag[k, oid] = -1e9
            h = h + mask_diag
            log_p = F.log_softmax(h, dim=1)
            loss = -(F_ij_d[bo] * log_p).sum() / max(F_train_total, 1.0)
            loss.backward(); opt.step()
            ep_loss += float(loss.item()); n_b += 1
        sched.step()
        model.eval()
        with torch.no_grad():
            v_loss = 0.0
            for b0 in range(0, len(val_idx), batch_size):
                bo = val_idx[b0:b0+batch_size]
                h = model.forward_block(static_d[bo], static_d, log_d_ij_d[bo])
                mask_diag = torch.zeros_like(h)
                for k, oid in enumerate(bo):
                    mask_diag[k, oid] = -1e9
                h = h + mask_diag
                log_p = F.log_softmax(h, dim=1)
                v_loss += float(-(F_ij_d[bo] * log_p).sum().item()) / max(F_val_total, 1.0)
        if v_loss < best_val - 1e-6:
            best_val = v_loss
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
            no_improve = 0
        else:
            no_improve += 1
        if ep == 0 or ep % verbose_every == 0 or ep == epochs - 1:
            print(f'  [DG s{seed}] ep {ep:4d}  train_nll {ep_loss/n_b:.4f}  val_nll {v_loss:.4f}  lr {opt.param_groups[0]["lr"]:.5f}  no_impr {no_improve}/{early_stop_patience}')
        if no_improve >= early_stop_patience:
            print(f'  [DG s{seed}] early stop at epoch {ep} (best val {best_val:.4f})')
            break
    if best_state is not None:
        model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        F_pred = torch.zeros_like(F_ij_d)
        for b0 in range(0, N, batch_size):
            bo = torch.arange(b0, min(b0+batch_size, N), device=dev)
            h = model.forward_block(static_d[bo], static_d, log_d_ij_d[bo])
            mask_diag = torch.zeros_like(h)
            for k, oid in enumerate(bo):
                mask_diag[k, oid] = -1e9
            h = h + mask_diag
            P = F.softmax(h, dim=1)
            F_pred[bo] = O_i[bo] * P
    return F_pred.cpu(), best_val, ep

dg_cpcs = []
dg_rows = []
for seed in range(3):
    print(f'\n--- seed {seed} ---')
    t0 = time.time()
    F_pred, best_val, last_ep = fit_deep_gravity(
        F_ij, static, log_d, train_mask, val_mask,
        epochs=1000, lr=1e-3, hidden=128, n_layers=5,
        seed=seed, batch_size=128, early_stop_patience=40,
        device=DEVICE, verbose_every=25,
    )
    dg_cpc = cpc(F_obs_np, F_pred.numpy(), val_mask_np)
    dg_cpcs.append(dg_cpc)
    dt = time.time() - t0
    print(f'  Deep Gravity seed={seed}: val_CPC(spatial) = {dg_cpc:.4f}  best_val_NLL={best_val:.4f}  epochs={last_ep}  t={dt:.1f}s')
    dg_rows.append({'seed': seed, 'val_cpc_spatial': dg_cpc, 'best_val_nll': best_val, 'epochs_run': last_ep, 'seconds': dt})

In [ ]:
# === Cell 7: assemble leaderboard ===
rows = []
rows.append({'model': 'gravity', 'seed': 'n/a', 'val_cpc_spatial': grav_cpc, **info_grav})
rows.append({'model': 'radiation', 'seed': 'n/a', 'val_cpc_spatial': rad_cpc})
for r in dg_rows:
    rows.append({'model': 'deep_gravity', **r})
df_out = pd.DataFrame(rows)
df_out.to_csv('baselines_spatial_holdout.csv', index=False)
print('Saved baselines_spatial_holdout.csv')
print()
print(df_out.to_string(index=False))

dg_mean = float(np.mean(dg_cpcs)); dg_std = float(np.std(dg_cpcs))
phase_b_v6_cpc = 0.2892   # from local spatial_holdout.json
print(f'\n=== leaderboard ===')
print(f'  Gravity (Wilson)     {grav_cpc:.4f}')
print(f'  Radiation (Simini12) {rad_cpc:.4f}')
print(f'  Deep Gravity (MLP)   {dg_mean:.4f} +- {dg_std:.4f}  (n=3)')
print(f'  Phase B v6 (ours)    {phase_b_v6_cpc:.4f}  (from local run)')
summary = {
    'heldout_boroughs': HELDOUT_BOROUGHS,
    'rows': rows,
    'leaderboard': {
        'gravity': grav_cpc, 'radiation': rad_cpc,
        'deep_gravity_mean': dg_mean, 'deep_gravity_std': dg_std,
        'phase_b_v6_local': phase_b_v6_cpc,
    },
    'device': DEVICE,
}
with open('baselines_spatial_holdout.json', 'w') as f:
    json.dump(summary, f, indent=2, default=float)
print('Saved baselines_spatial_holdout.json')

In [ ]:
# === Cell 8: download results ===
from google.colab import files
files.download('baselines_spatial_holdout.csv')
files.download('baselines_spatial_holdout.json')